In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
# Ensure PySpark can find Homebrew Java inside this notebook kernel.
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"
repo_root = Path.cwd().parent
output_loc = str(repo_root/"yellowline"/"data"/"output")

silver_path = f'{output_loc}/silver'

spark = (
    SparkSession.builder
    .appName("notebook-spark-session")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
    .getOrCreate()
)

silver_df = spark.read.parquet(silver_path)
silver_df.show(5, truncate=False)
silver_df.createOrReplaceTempView("silver_temp")


+--------+--------------------+---------------------+---------------+-------------+------------+------------+--------------+-----------+---------------+------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|pickup_borough|pickup_zone|dropoff_borough|dropoff_zone|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+--------------+-----------+---------------+------------+-----------+----------+------------+
|2       |2015-01-20 11:44:14 |2015-01-20 11:52:17  |1              |1.54         |NULL        |NULL        |NULL          |NULL       |NULL           |NULL        |7.5        |1.66      |9.96        |
|2       |2015-01-20 11:44:18 |2015-01-20 11:55:26  |2              |1.13         |NULL        |NULL        |NULL          |NULL       |NULL           |NULL        |8.5        |1.7       |11.0

In [20]:
res = spark.sql("SELECT * FROM {silver_df} WHERE PULocationID is not NULL AND DOLocationID is not NULL")

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '{'. SQLSTATE: 42601 (line 1, pos 14)

== SQL ==
SELECT * FROM {silver_df} WHERE PULocationID is not NULL AND DOLocationID is not NULL
--------------^^^


In [8]:
bronze_path = f'{output_loc}/bronze'
bronze_df = spark.read.parquet(bronze_path)
bronze_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-11 23:26:53|  2015-01-11 23:28:09|              3|         0.28|      NULL|                 N|        NULL|        NULL|           1|        3.0|  0.5|    0.5|       0.0|         0.0|                  0.3|         4.3|
|       2| 2015-01-11 23:26:53|  2015-01-11 23:3

In [17]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType, StructField
TRIP_SCHEMA = StructType(
    [
        StructField("VendorID", IntegerType(), True),
        StructField(
            "tpep_pickup_datetime", StringType(), True
        ),  # cast to timestamp in Silver
        StructField("tpep_dropoff_datetime", StringType(), True),
        StructField("passenger_count", IntegerType(), True),
        StructField("trip_distance", DoubleType(), True),
        StructField("RatecodeID", IntegerType(), True),
        StructField("store_and_fwd_flag", StringType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", IntegerType(), True),
        StructField("payment_type", IntegerType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("extra", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("improvement_surcharge", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
    ]
)

raw_df = spark.read.format('csv').schema(TRIP_SCHEMA).load(str(repo_root)+ "/yellowline/data/raw/yellow_tripdata.csv")
raw_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|      fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------------+-----+-------+----------+------------+---------------------+------------+
|    NULL|tpep_pickup_datetime| tpep_dropoff_date...|           NULL|         NULL|      NULL|   pickup_latitude|        NULL|        NULL|        NULL|             NULL| NULL|   NULL|      NULL|        NULL|                 NULL|        NULL|
|       2| 2015-01-15 19